# Sampled Scatter Decoding Orchestrator — Quality-Filtered Speech

Identical to `scat_classifier_sampled_nocv.ipynb` but restricted to words that pass
quality filtration (`{patient}_word_df_filtered.csv`).

No speaker restriction is applied — all quality-filtered words (patient + non-patient)
are used.

Word subset indices are saved per-patient and passed to the worker via `--word-idx-path`.

In [1]:
import json
import subprocess
from pathlib import Path

import dill as pickle
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)

In [2]:
# -- Config -----------------------------------------------------------------
PATIENTS = ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']

VAD_ROOT    = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new')
LEGACY_ROOT = Path('/mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_out')

RUN_NAME = 'scat_xgboost_sampled_norm_filtered'

CLUSTER_PREDS_OVERRIDE = {}
FRS_PATH_OVERRIDE      = {}

N_RESAMPLES  = 20
FIRST_STAGE  = [0, 1, 2]
SECOND_STAGE = list(range(3, N_RESAMPLES))
SEED_STRIDE  = 42

PYTHON_BIN  = '/scratch/tahaismail424/miniforge3/envs/speech_247_env/bin/python3'
WORKER_PATH = Path('/scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py')

PARTITION = 'Universal'
QOS       = 'default_tier'
CPUS      = 8
MEM       = '40G'
TIME      = '48:00:00'

TEST_SIZE    = 0.2
SEARCH_ITERS = 40
CV_SPLITS    = 4
N_SHUFFLES   = 50
SCORING      = 'f1_macro'
PERM_METRIC  = 'f1_macro'

print('patients:', PATIENTS)
print('worker:', WORKER_PATH)

patients: ['YEY', 'YEZ', 'YFA', 'YFB', 'YFC', 'YFD', 'YFE', 'YFF', 'YFG', 'YFI', 'YFK', 'YFL', 'YFM', 'YFN', 'YFP', 'YFQ', 'YFR', 'YFT', 'YFU', 'YFV']
worker: /scratch/tahaismail424/speech_247/notebooks/REBIRTH_2!/standard_decoding_analysis/scat_classifier_worker.py


### Compute & Save Quality-Filtered Word Indices Per Patient

In [3]:
def compute_filtered_indices(patient: str) -> 'Path | None':
    patient_root = VAD_ROOT / patient
    filtered_csv = patient_root / f'{patient}_word_df_filtered.csv'
    idx_out_path = patient_root / 'standard_decoding_analysis' / RUN_NAME / f'{patient}_filtered_word_idx.npy'

    if not filtered_csv.exists():
        print(f'  [{patient}] missing {filtered_csv.name}')
        return None

    filtered_df = pd.read_csv(filtered_csv)
    word_idx = filtered_df['original_word_idx'].values.astype(np.int64)

    idx_out_path.parent.mkdir(parents=True, exist_ok=True)
    np.save(idx_out_path, word_idx)

    print(f'  [{patient}] {len(word_idx):,} quality-filtered words → {idx_out_path.name}')
    return idx_out_path


print(f'Computing filtered word indices for {len(PATIENTS)} patients...')
patient_idx_paths = {}
for patient in PATIENTS:
    idx_path = compute_filtered_indices(patient)
    if idx_path is not None:
        patient_idx_paths[patient] = idx_path

print(f'\nReady: {len(patient_idx_paths)} / {len(PATIENTS)} patients')

Computing filtered word indices for 20 patients...
  [YEY] 40,351 quality-filtered words → YEY_filtered_word_idx.npy
  [YEZ] 71,809 quality-filtered words → YEZ_filtered_word_idx.npy
  [YFA] 521,360 quality-filtered words → YFA_filtered_word_idx.npy
  [YFB] 128,315 quality-filtered words → YFB_filtered_word_idx.npy
  [YFC] 292,428 quality-filtered words → YFC_filtered_word_idx.npy
  [YFD] 388,734 quality-filtered words → YFD_filtered_word_idx.npy
  [YFE] 199,717 quality-filtered words → YFE_filtered_word_idx.npy
  [YFF] 228,795 quality-filtered words → YFF_filtered_word_idx.npy
  [YFG] 48,567 quality-filtered words → YFG_filtered_word_idx.npy
  [YFI] 126,297 quality-filtered words → YFI_filtered_word_idx.npy
  [YFK] 372,685 quality-filtered words → YFK_filtered_word_idx.npy
  [YFL] 90,616 quality-filtered words → YFL_filtered_word_idx.npy
  [YFM] 216,077 quality-filtered words → YFM_filtered_word_idx.npy
  [YFN] 286,018 quality-filtered words → YFN_filtered_word_idx.npy
  [YFP] 375,545

In [4]:
# -- Path resolution helpers -------------------------------------------------
def first_existing(paths):
    for p in paths:
        if p is not None and Path(p).exists():
            return Path(p)
    return None


def patient_root(patient: str) -> Path:
    return VAD_ROOT / patient


def output_root(patient: str) -> Path:
    out = patient_root(patient) / 'standard_decoding_analysis' / RUN_NAME
    out.mkdir(parents=True, exist_ok=True)
    return out


def logs_dir(patient: str) -> Path:
    p = patient_root(patient) / 'decoding_slurm_logs' / RUN_NAME
    p.mkdir(parents=True, exist_ok=True)
    return p


def scripts_dir(patient: str) -> Path:
    p = patient_root(patient) / 'decoding_slurm_scripts' / RUN_NAME
    p.mkdir(parents=True, exist_ok=True)
    return p


def resolve_patient_inputs(patient: str) -> dict:
    root = patient_root(patient)
    cluster_override = CLUSTER_PREDS_OVERRIDE.get(patient)
    frs_override     = FRS_PATH_OVERRIDE.get(patient)

    new_clusters = first_existing([root / 'embeddings' / 'semantic_cluster_predictions.npy'])
    new_frs = first_existing([
        root / 'neural_embeddings' / 'word_frs.npy',
        root / 'all_convo_recording' / 'word_avg_frs.npy',
    ])
    legacy_frs = first_existing([LEGACY_ROOT / patient / 'all_convo_recording' / 'word_avg_frs.npy'])

    if cluster_override is not None or frs_override is not None:
        cluster_preds_path = first_existing([cluster_override, new_clusters])
        frs_path           = first_existing([frs_override, new_frs, legacy_frs])
        input_source = 'override'
    elif new_clusters is not None and new_frs is not None:
        cluster_preds_path = new_clusters
        frs_path           = new_frs
        input_source = 'vad_new'
    else:
        cluster_preds_path = None
        frs_path           = None
        input_source = 'missing'

    word_idx_path = patient_idx_paths.get(patient)
    out = output_root(patient)
    return {
        'patient': patient,
        'patient_root': root,
        'input_source': input_source,
        'cluster_preds_path': cluster_preds_path,
        'frs_path': frs_path,
        'word_idx_path': word_idx_path,
        'output_root': out,
        'consensus_params_json': out / 'consensus_best_params.json',
        'aggregate_csv': out / 'scat_sampled_all_resamples.csv',
        'aggregate_pkl': out / 'scat_sampled_all_resamples.pkl',
        'logs_dir': logs_dir(patient),
        'scripts_dir': scripts_dir(patient),
    }


def choose_consensus_params(param_options: list) -> dict:
    params = dict(param_options[0])
    params['max_depth']        = int(np.min([p['max_depth'] for p in param_options]))
    params['min_child_weight'] = int(np.max([p['min_child_weight'] for p in param_options]))
    params['gamma']            = float(np.max([p['gamma'] for p in param_options]))
    params['reg_lambda']       = float(np.max([p['reg_lambda'] for p in param_options]))
    params['reg_alpha']        = float(np.max([p['reg_alpha'] for p in param_options]))
    params['subsample']        = float(np.median([p['subsample'] for p in param_options]))
    params['colsample_bytree'] = float(np.median([p['colsample_bytree'] for p in param_options]))
    params['learning_rate']    = float(np.median([p['learning_rate'] for p in param_options]))
    params['n_estimators']     = int(np.median([p['n_estimators'] for p in param_options]))
    params['tree_method'] = 'hist'
    params['device'] = 'cpu'
    params.pop('predictor', None)
    params.pop('use_label_encoder', None)
    return params

In [5]:
# -- Status helpers ----------------------------------------------------------
def resample_paths(out_root: Path, r: int) -> dict:
    return {
        'pkl':              out_root / f'scat_sampled_resample_{r}_.pkl',
        'summary_json':     out_root / f'summary_resample_{r}.json',
        'best_params_json': out_root / f'best_params_resample_{r}.json',
        'success':          out_root / f'resample_{r}_SUCCESS',
        'error':            out_root / f'resample_{r}_error.txt',
    }


def build_patient_status(patient: str) -> pd.DataFrame:
    info = resolve_patient_inputs(patient)
    rows = []
    for r in range(N_RESAMPLES):
        paths = resample_paths(info['output_root'], r)
        rows.append({
            'patient': patient,
            'resample_idx': r,
            'stage': 'hyperparam' if r in FIRST_STAGE else 'fixed',
            'seed': r * SEED_STRIDE,
            'cluster_preds_path':    info['cluster_preds_path'],
            'frs_path':              info['frs_path'],
            'word_idx_path':         info['word_idx_path'],
            'has_inputs':            info['cluster_preds_path'] is not None and info['frs_path'] is not None and info['word_idx_path'] is not None,
            'consensus_params_json': info['consensus_params_json'],
            'consensus_ready':       info['consensus_params_json'].exists(),
            'pkl_path':              paths['pkl'],
            'summary_json':          paths['summary_json'],
            'best_params_json':      paths['best_params_json'],
            'success_path':          paths['success'],
            'error_path':            paths['error'],
            'has_pkl':               paths['pkl'].exists(),
            'has_summary':           paths['summary_json'].exists(),
            'has_best_params':       paths['best_params_json'].exists(),
            'has_success':           paths['success'].exists(),
            'has_error':             paths['error'].exists(),
        })
    df = pd.DataFrame(rows)
    first_three_done = bool(df[df['resample_idx'].isin(FIRST_STAGE)]['has_success'].all()) if not df.empty else False
    df['first_three_done'] = first_three_done
    df['ready_phase1'] = df['has_inputs'] & df['resample_idx'].isin(FIRST_STAGE) & ~df['has_success']
    df['ready_phase2'] = df['has_inputs'] & df['resample_idx'].isin(SECOND_STAGE) & first_three_done & df['consensus_ready'] & ~df['has_success']
    return df


status_df = pd.concat([build_patient_status(pt) for pt in PATIENTS], ignore_index=True)
status_df[['patient','resample_idx','stage','has_inputs','has_success','has_error','has_best_params','consensus_ready','ready_phase1','ready_phase2']]

,patient,resample_idx,stage,has_inputs,has_success,has_error,has_best_params,consensus_ready,ready_phase1,ready_phase2
0,YEY,0,hyperparam,True,True,False,True,True,False,False
1,YEY,1,hyperparam,True,True,False,True,True,False,False
2,YEY,2,hyperparam,True,True,False,True,True,False,False
3,YEY,3,fixed,True,True,False,True,True,False,False
4,YEY,4,fixed,True,True,False,True,True,False,False
...,...,...,...,...,...,...,...,...,...,...
395,YFV,15,fixed,True,True,False,True,True,False,False
396,YFV,16,fixed,True,True,False,True,True,False,False
397,YFV,17,fixed,True,True,False,True,True,False,False
398,YFV,18,fixed,True,True,False,True,True,False,False


In [6]:
# -- Patient input summary ---------------------------------------------------
patient_info_df = pd.DataFrame([resolve_patient_inputs(pt) for pt in PATIENTS])
patient_info_df[['patient', 'input_source', 'cluster_preds_path', 'frs_path', 'word_idx_path', 'output_root']]

,patient,input_source,cluster_preds_path,frs_path,word_idx_path,output_root
0,YEY,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
1,YEZ,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
2,YFA,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
3,YFB,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
4,YFC,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
5,YFD,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
6,YFE,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
7,YFF,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
8,YFG,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...
9,YFI,vad_new,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...,/mnt/labworlds/Hayden/Hayden_Lab/speech_247/va...


In [8]:
# -- Submit first 3 resamples per patient -----------------------------------
DRY_RUN    = False
SUBMIT_ONE = False

submitted = []
failed    = []

for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    for _, row in patient_status[patient_status['ready_phase1']].iterrows():
        r = int(row['resample_idx'])
        word_idx_arg = f'--word-idx-path "{info["word_idx_path"]}"' if info['word_idx_path'] else ''
        sbatch_text = f'''#!/bin/bash
#SBATCH --job-name=scat_fs_{patient}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={info['logs_dir']}/{patient}_r{r}_%j.out
#SBATCH --error={info['logs_dir']}/{patient}_r{r}_%j.err

set -eo pipefail

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH}   --patient {patient}   --resample-idx {r}   --cluster-preds-path "{info['cluster_preds_path']}"   --frs-path "{info['frs_path']}"   --out-dir "{info['output_root']}"   {word_idx_arg}   --test-size {TEST_SIZE}   --n-iter {SEARCH_ITERS}   --cv-splits {CV_SPLITS}   --n-shuffles {N_SHUFFLES}   --seed-stride {SEED_STRIDE}   --n-jobs {CPUS * 2}   --scoring {SCORING}   --perm-metric {PERM_METRIC}

echo "END: $(date)"
'''
        sbatch_path = info['scripts_dir'] / f'{patient}_resample_{r}.sbatch'
        sbatch_path.write_text(sbatch_text)
        if DRY_RUN:
            submitted.append((patient, r, 'dry-run'))
            print(f'dry-run: {patient} r={r}')
        else:
            try:
                res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
                submitted.append((patient, r, res.stdout.strip()))
                print(f'submitted: {patient} r={r} -> {res.stdout.strip()}')
            except subprocess.CalledProcessError as exc:
                failed.append((patient, r, exc.stderr.strip()))
                print(f'FAILED SUBMISSION: {patient} r={r}\n{exc.stderr}')
        if SUBMIT_ONE and submitted:
            break
    if SUBMIT_ONE and submitted:
        break

print(f'submitted={len(submitted)}, failed={len(failed)}')

submitted: YEY r=0 -> Submitted batch job 224876
submitted: YEY r=1 -> Submitted batch job 224877
submitted: YEY r=2 -> Submitted batch job 224878
submitted: YEZ r=0 -> Submitted batch job 224879
submitted: YEZ r=1 -> Submitted batch job 224880
submitted: YEZ r=2 -> Submitted batch job 224881
submitted: YFA r=0 -> Submitted batch job 224882
submitted: YFA r=1 -> Submitted batch job 224883
submitted: YFA r=2 -> Submitted batch job 224884
submitted=9, failed=0


In [7]:
# -- Build consensus params once resamples 0..2 finish ----------------------
built   = []
blocked = []
for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    if info['consensus_params_json'].exists():
        print(f'consensus already exists for {patient}: {info["consensus_params_json"]}')
        continue
    first_three = patient_status[patient_status['resample_idx'].isin(FIRST_STAGE)]
    if not bool(first_three['has_success'].all()):
        blocked.append(patient)
        print(f'waiting on first three for {patient}')
        continue
    param_options = []
    for r in FIRST_STAGE:
        path = info['output_root'] / f'best_params_resample_{r}.json'
        param_options.append(json.loads(path.read_text()))
    consensus = choose_consensus_params(param_options)
    info['consensus_params_json'].write_text(json.dumps(consensus, indent=2))
    built.append(patient)
    print(f'built consensus params for {patient} -> {info["consensus_params_json"]}')

print('built:', built)
print('blocked:', blocked)

consensus already exists for YEY: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEY/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/consensus_best_params.json
consensus already exists for YEZ: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEZ/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/consensus_best_params.json
consensus already exists for YFA: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/consensus_best_params.json
consensus already exists for YFB: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFB/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/consensus_best_params.json
consensus already exists for YFC: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFC/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/consensus_best_params.json
consensus already exists for YFD: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFD/standard_decodi

In [8]:
# -- Submit remaining 17 resamples after consensus exists -------------------
DRY_RUN    = False
SUBMIT_ONE = False

submitted = []
failed    = []

for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    for _, row in patient_status[patient_status['ready_phase2']].iterrows():
        r = int(row['resample_idx'])
        word_idx_arg = f'--word-idx-path "{info["word_idx_path"]}"' if info['word_idx_path'] else ''
        sbatch_text = f'''#!/bin/bash
#SBATCH --job-name=scat_fs_{patient}_{r}
#SBATCH --partition={PARTITION}
#SBATCH --qos={QOS}
#SBATCH --cpus-per-task={CPUS}
#SBATCH --mem={MEM}
#SBATCH --time={TIME}
#SBATCH --output={info['logs_dir']}/{patient}_r{r}_%j.out
#SBATCH --error={info['logs_dir']}/{patient}_r{r}_%j.err

set -eo pipefail

echo "HOSTNAME: $(hostname)"
echo "START: $(date)"
echo "PATIENT: {patient}"
echo "RESAMPLE: {r}"

{PYTHON_BIN} {WORKER_PATH}   --patient {patient}   --resample-idx {r}   --cluster-preds-path "{info['cluster_preds_path']}"   --frs-path "{info['frs_path']}"   --out-dir "{info['output_root']}"   --params-json "{info['consensus_params_json']}"   {word_idx_arg}   --test-size {TEST_SIZE}   --n-iter {SEARCH_ITERS}   --cv-splits {CV_SPLITS}   --n-shuffles {N_SHUFFLES}   --seed-stride {SEED_STRIDE}   --n-jobs {CPUS * 2}   --scoring {SCORING}   --perm-metric {PERM_METRIC}

echo "END: $(date)"
'''
        sbatch_path = info['scripts_dir'] / f'{patient}_resample_{r}.sbatch'
        sbatch_path.write_text(sbatch_text)
        if DRY_RUN:
            submitted.append((patient, r, 'dry-run'))
            print(f'dry-run: {patient} r={r}')
        else:
            try:
                res = subprocess.run(['sbatch', str(sbatch_path)], capture_output=True, text=True, check=True)
                submitted.append((patient, r, res.stdout.strip()))
                print(f'submitted: {patient} r={r} -> {res.stdout.strip()}')
            except subprocess.CalledProcessError as exc:
                failed.append((patient, r, exc.stderr.strip()))
                print(f'FAILED SUBMISSION: {patient} r={r}\n{exc.stderr}')
        if SUBMIT_ONE and submitted:
            break
    if SUBMIT_ONE and submitted:
        break

print(f'submitted={len(submitted)}, failed={len(failed)}')

submitted=0, failed=0


In [9]:
# -- Error inspection --------------------------------------------------------
for patient in PATIENTS:
    patient_status = build_patient_status(patient)
    errs = patient_status[patient_status['has_error']]
    if errs.empty:
        print(f'no error files for {patient}')
        continue
    print(f'errors for {patient}:')
    display(errs[['resample_idx', 'error_path']])

no error files for YEY
no error files for YEZ
no error files for YFA
no error files for YFB
no error files for YFC
no error files for YFD
no error files for YFE
no error files for YFF
no error files for YFG
no error files for YFI
no error files for YFK
no error files for YFL
no error files for YFM
no error files for YFN
no error files for YFP
no error files for YFQ
no error files for YFR
no error files for YFT
no error files for YFU
no error files for YFV


In [10]:
# -- Aggregate full patient results once all 20 finish ----------------------
for patient in PATIENTS:
    info = resolve_patient_inputs(patient)
    patient_status = build_patient_status(patient)
    if not bool(patient_status['has_success'].all()):
        print(f'skipping aggregate for {patient}: not all resamples finished')
        continue

    summary_rows = []
    aggregate    = {}
    for r in range(N_RESAMPLES):
        with open(info['output_root'] / f'summary_resample_{r}.json') as f:
            summary_rows.append(json.load(f))
        with open(info['output_root'] / f'scat_sampled_resample_{r}_.pkl', 'rb') as f:
            aggregate[f'resample_{r}'] = pickle.load(f)

    summary_df = pd.DataFrame(summary_rows).sort_values('resample_idx').reset_index(drop=True)
    summary_df.to_csv(info['aggregate_csv'], index=False)
    with open(info['aggregate_pkl'], 'wb') as f:
        pickle.dump(aggregate, f)

    print(f'aggregated {patient}')
    print(f"  csv: {info['aggregate_csv']}")
    print(f"  pkl: {info['aggregate_pkl']}")
    display(summary_df)

aggregated YEY
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEY/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEY/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YEY,0,0,8763,8763,1753,0.175128,0.138760,0.136723,0.175128,0.565047,0.119186,0.136338,5.378170e-12,f1_macro,0.136723,0.019608,0.132951,shuffle_training_X_only
1,YEY,1,42,8763,8763,1753,0.179122,0.141564,0.138558,0.179122,0.560683,0.119186,0.136338,1.978836e-13,f1_macro,0.138558,0.019608,0.130435,shuffle_training_X_only
2,YEY,2,84,8763,8763,1753,0.176840,0.137614,0.132767,0.176840,0.556688,0.119186,0.136338,1.336448e-12,f1_macro,0.132767,0.019608,0.131531,shuffle_training_X_only
3,YEY,3,126,8763,8763,1753,0.160867,0.123174,0.116239,0.160867,0.558047,0.119186,0.136338,1.490749e-07,f1_macro,0.116239,0.019608,NaN,shuffle_training_X_only
4,YEY,4,168,8763,8763,1753,0.171135,0.135722,0.133355,0.171135,0.556481,0.119186,0.136338,1.209803e-10,f1_macro,0.133355,0.019608,NaN,shuffle_training_X_only
5,YEY,5,210,8763,8763,1753,0.162578,0.126910,0.121809,0.162578,0.543433,0.119186,0.136338,4.983278e-08,f1_macro,0.121809,0.019608,NaN,shuffle_training_X_only
6,YEY,6,252,8763,8763,1753,0.183115,0.141205,0.132905,0.183115,0.558324,0.119186,0.136338,6.043418e-15,f1_macro,0.132905,0.019608,NaN,shuffle_training_X_only
7,YEY,7,294,8763,8763,1753,0.172276,0.132973,0.126980,0.172276,0.568275,0.119186,0.136338,5.068293e-11,f1_macro,0.126980,0.019608,NaN,shuffle_training_X_only
8,YEY,8,336,8763,8763,1753,0.181403,0.140211,0.134366,0.181403,0.556996,0.119186,0.136338,2.757089e-14,f1_macro,0.134366,0.019608,NaN,shuffle_training_X_only
9,YEY,9,378,8763,8763,1753,0.166001,0.127925,0.121423,0.166001,0.565774,0.119186,0.136338,4.993013e-09,f1_macro,0.121423,0.019608,NaN,shuffle_training_X_only


aggregated YEZ
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEZ/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YEZ/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YEZ,0,0,17392,17392,3479,0.208106,0.148873,0.141016,0.208106,0.599677,0.127479,0.146306,3.032064e-40,f1_macro,0.141016,0.019608,0.139435,shuffle_training_X_only
1,YEZ,1,42,17392,17392,3479,0.204369,0.147069,0.138614,0.204369,0.612032,0.127479,0.146306,5.588587e-37,f1_macro,0.138614,0.019608,0.134523,shuffle_training_X_only
2,YEZ,2,84,17392,17392,3479,0.186835,0.134598,0.129252,0.186835,0.582650,0.127479,0.146306,2.095565e-23,f1_macro,0.129252,0.019608,0.144972,shuffle_training_X_only
3,YEZ,3,126,17392,17392,3479,0.211842,0.151479,0.143197,0.211842,0.606871,0.127479,0.146306,1.227428e-43,f1_macro,0.143197,0.019608,NaN,shuffle_training_X_only
4,YEZ,4,168,17392,17392,3479,0.208393,0.153379,0.146963,0.208393,0.605020,0.127479,0.146306,1.679782e-40,f1_macro,0.146963,0.019608,NaN,shuffle_training_X_only
5,YEZ,5,210,17392,17392,3479,0.208968,0.150838,0.142987,0.208968,0.606155,0.127479,0.146306,5.128935e-41,f1_macro,0.142987,0.019608,NaN,shuffle_training_X_only
6,YEZ,6,252,17392,17392,3479,0.205519,0.147652,0.139241,0.205519,0.604616,0.127479,0.146306,5.703698e-38,f1_macro,0.139241,0.019608,NaN,shuffle_training_X_only
7,YEZ,7,294,17392,17392,3479,0.211268,0.149128,0.136406,0.211268,0.596780,0.127479,0.146306,4.160735e-43,f1_macro,0.136406,0.019608,NaN,shuffle_training_X_only
8,YEZ,8,336,17392,17392,3479,0.195458,0.141455,0.134578,0.195458,0.592761,0.127479,0.146306,1.026535e-29,f1_macro,0.134578,0.019608,NaN,shuffle_training_X_only
9,YEZ,9,378,17392,17392,3479,0.213567,0.154214,0.146082,0.213567,0.609714,0.127479,0.146306,3.024557e-45,f1_macro,0.146082,0.019608,NaN,shuffle_training_X_only


aggregated YFA
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFA/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/scat_sampled_all_resamples.pkl


...

,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFT,0,0,108967,100268,20054,0.214371,0.163626,0.150745,0.214371,0.625058,0.119738,0.137878,9.607208e-312,f1_macro,0.150745,0.019608,0.150128,shuffle_training_X_only
1,YFT,1,42,108967,100254,20051,0.218892,0.168799,0.158766,0.218892,0.622429,0.119733,0.137499,0.000000e+00,f1_macro,0.158766,0.019608,0.152831,shuffle_training_X_only
2,YFT,2,84,108967,100285,20057,0.219125,0.167508,0.154780,0.219125,0.625957,0.119751,0.137807,0.000000e+00,f1_macro,0.154780,0.019608,0.153120,shuffle_training_X_only
3,YFT,3,126,108967,100169,20034,0.220176,0.169601,0.159409,0.220176,0.625244,0.119712,0.137816,0.000000e+00,f1_macro,0.159409,0.019608,NaN,shuffle_training_X_only
4,YFT,4,168,108967,100253,20051,0.212259,0.162417,0.150913,0.212259,0.617950,0.119732,0.137799,4.810950e-299,f1_macro,0.150913,0.019608,NaN,shuffle_training_X_only
5,YFT,5,210,108967,100184,20037,0.217647,0.166073,0.153119,0.217647,0.625006,0.119709,0.137745,0.000000e+00,f1_macro,0.153119,0.019608,NaN,shuffle_training_X_only
6,YFT,6,252,108967,100190,20038,0.221579,0.169111,0.155965,0.221579,0.621793,0.119710,0.137539,0.000000e+00,f1_macro,0.155965,0.019608,NaN,shuffle_training_X_only
7,YFT,7,294,108967,100199,20040,0.218713,0.168056,0.157107,0.218713,0.626817,0.119713,0.137725,0.000000e+00,f1_macro,0.157107,0.019608,NaN,shuffle_training_X_only
8,YFT,8,336,108967,100191,20039,0.214482,0.163944,0.151987,0.214482,0.620924,0.119712,0.137532,2.166112e-312,f1_macro,0.151987,0.019608,NaN,shuffle_training_X_only
9,YFT,9,378,108967,100241,20049,0.219462,0.168072,0.156271,0.219462,0.623440,0.119730,0.137812,0.000000e+00,f1_macro,0.156271,0.019608,NaN,shuffle_training_X_only


aggregated YFU
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFU/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFU/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFU,0,0,101952,99324,19865,0.219179,0.167315,0.150475,0.219179,0.622265,0.118710,0.136924,0.000000e+00,f1_macro,0.150475,0.019608,0.148808,shuffle_training_X_only
1,YFU,1,42,101952,99337,19868,0.214818,0.166074,0.153832,0.214818,0.618827,0.118706,0.136753,1.134523e-319,f1_macro,0.153832,0.019608,0.149857,shuffle_training_X_only
2,YFU,2,84,101952,99340,19868,0.216378,0.166161,0.151495,0.216378,0.622356,0.118716,0.136853,0.000000e+00,f1_macro,0.151495,0.019608,0.153002,shuffle_training_X_only
3,YFU,3,126,101952,99278,19856,0.217869,0.167844,0.153779,0.217869,0.617612,0.118685,0.136835,0.000000e+00,f1_macro,0.153779,0.019608,NaN,shuffle_training_X_only
4,YFU,4,168,101952,99292,19859,0.216778,0.166297,0.151936,0.216778,0.619807,0.118700,0.136865,0.000000e+00,f1_macro,0.151936,0.019608,NaN,shuffle_training_X_only
5,YFU,5,210,101952,99331,19867,0.215282,0.165303,0.150539,0.215282,0.619028,0.118705,0.136860,1.630417e-322,f1_macro,0.150539,0.019608,NaN,shuffle_training_X_only
6,YFU,6,252,101952,99307,19862,0.217652,0.167026,0.152300,0.217652,0.620073,0.118705,0.136945,0.000000e+00,f1_macro,0.152300,0.019608,NaN,shuffle_training_X_only
7,YFU,7,294,101952,99302,19861,0.220331,0.169079,0.154410,0.220331,0.621137,0.118703,0.136801,0.000000e+00,f1_macro,0.154410,0.019608,NaN,shuffle_training_X_only
8,YFU,8,336,101952,99293,19859,0.218490,0.167885,0.153106,0.218490,0.617699,0.118700,0.136915,0.000000e+00,f1_macro,0.153106,0.019608,NaN,shuffle_training_X_only
9,YFU,9,378,101952,99309,19862,0.215134,0.165410,0.151598,0.215134,0.620197,0.118705,0.136844,1.590891e-321,f1_macro,0.151598,0.019608,NaN,shuffle_training_X_only


aggregated YFV
  csv: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFV/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/scat_sampled_all_resamples.csv
  pkl: /mnt/labworlds/Hayden/Hayden_Lab/speech_247/vad_new/YFV/standard_decoding_analysis/scat_xgboost_sampled_norm_filtered/scat_sampled_all_resamples.pkl


,patient,resample_idx,seed,n_sampled_raw,n_sampled_final,n_test,acc,balanced_accuracy,f1_macro,f1_micro,auc_macro_ovr,baseline_proportional_acc,baseline_majority_acc,binomial_p_vs_proportional,perm_metric,observed_perm_metric,perm_pvalue,cv_best_score,perm_null
0,YFV,0,0,29997,26531,5307,0.209723,0.160058,0.149755,0.209723,0.620381,0.120720,0.141511,7.179916e-75,f1_macro,0.149755,0.019608,0.152109,shuffle_training_X_only
1,YFV,1,42,29997,26498,5300,0.210000,0.159024,0.150441,0.210000,0.609651,0.120752,0.141698,3.905417e-75,f1_macro,0.150441,0.019608,0.150072,shuffle_training_X_only
2,YFV,2,84,29997,26443,5289,0.205521,0.156008,0.147578,0.205521,0.618796,0.120638,0.141993,1.601527e-68,f1_macro,0.147578,0.019608,0.154054,shuffle_training_X_only
3,YFV,3,126,29997,26488,5298,0.208003,0.157350,0.148566,0.208003,0.613497,0.120697,0.140808,3.380986e-72,f1_macro,0.148566,0.019608,NaN,shuffle_training_X_only
4,YFV,4,168,29997,26463,5293,0.205932,0.157769,0.150498,0.205932,0.613610,0.120698,0.142074,4.602782e-69,f1_macro,0.150498,0.019608,NaN,shuffle_training_X_only
5,YFV,5,210,29997,26476,5296,0.201284,0.156008,0.149057,0.201284,0.613128,0.120689,0.141427,1.887118e-62,f1_macro,0.149057,0.019608,NaN,shuffle_training_X_only
6,YFV,6,252,29997,26527,5306,0.209951,0.162439,0.156011,0.209951,0.611975,0.120717,0.141726,3.283800e-75,f1_macro,0.156011,0.019608,NaN,shuffle_training_X_only
7,YFV,7,294,29997,26519,5304,0.214367,0.164766,0.158187,0.214367,0.619751,0.120777,0.141780,6.294986e-82,f1_macro,0.158187,0.019608,NaN,shuffle_training_X_only
8,YFV,8,336,29997,26462,5293,0.207255,0.156917,0.147399,0.207255,0.608153,0.120666,0.142074,4.487400e-71,f1_macro,0.147399,0.019608,NaN,shuffle_training_X_only
9,YFV,9,378,29997,26486,5298,0.205927,0.156520,0.147617,0.205927,0.611452,0.120703,0.141563,4.135007e-69,f1_macro,0.147617,0.019608,NaN,shuffle_training_X_only
